# Olist E-Commerce Analysis
## Notebook 2 — Data Cleaning

Goals:
- Fix datetime columns in the orders table
- Handle nulls
- Merge key tables into one analysis-ready dataframe

## Setup
Loading the six core tables needed for the master dataframe. Geolocation and sellers are excluded from this analysis.

In [ ]:
import pandas as pd
data_path = "../data/raw/"

orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")

### Fix datetime columns
All timestamp columns are loaded as strings — converting to datetime for date math.

In [9]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### Create delivery time features
- `delivery_time_days` — how long the order actually took to arrive
- `delay_days` — difference between actual and estimated delivery (negative = early, positive = late)

In [10]:
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

orders[['order_id', 'delivery_time_days', 'delay_days']].head(10)

,order_id,delivery_time_days,delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0
5,a4591c265e18cb1dcee52889e2d8acc3,16.0,-6.0
6,136cce7faa42fdb2cefd53fdc79a6098,NaN,NaN
7,6514b8ad8028c9f2cc2374ded245783f,9.0,-12.0
8,76c6e866289321a7c93b82b54852dc33,9.0,-32.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,18.0,-7.0


Early results show most orders arriving ahead of schedule — Olist appears to set conservative delivery estimates.

### Filter to delivered orders only
Keeping only orders with status 'delivered' for delivery time analysis.

In [11]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Total orders: {len(orders)}")
print(f"Delivered orders: {len(orders_delivered)}")
print(f"Dropped: {len(orders) - len(orders_delivered)}")

Total orders: 99441
Delivered orders: 96478
Dropped: 2963


### Build master dataframe
Merging orders, order_items, payments, reviews, customers and products into one analysis-ready table.

In [12]:
df = orders_delivered.merge(order_items, on='order_id', how='left')
df = df.merge(order_payments, on='order_id', how='left')
df = df.merge(order_reviews, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(products, on='product_id', how='left')

print(df.shape)
df.head()

(115723, 38)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delay_days,...,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,sao paulo,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,barreiras,BA,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,vianopolis,GO,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0


### Checking nulls across the master dataframe

In [13]:
df.isnull().sum()[df.isnull().sum() > 0]

order_approved_at                    15
order_delivered_carrier_date          2
order_delivered_customer_date         8
delivery_time_days                    8
delay_days                            8
payment_sequential                    3
payment_type                          3
payment_installments                  3
payment_value                         3
review_id                           861
review_score                        861
review_comment_title             102139
review_comment_message            67628
review_creation_date                861
review_answer_timestamp             861
product_category_name              1638
product_name_lenght                1638
product_description_lenght         1638
product_photos_qty                 1638
product_weight_g                     20
product_length_cm                    20
product_height_cm                    20
product_width_cm                     20
dtype: int64

**Key observations:**
- `review_score` has 861 nulls — rows without a review, will be dropped as review score is central to the analysis
- `review_comment_title` and `review_comment_message` have high nulls — expected, most customers only leave a star rating
- `product_category_name` has 1,638 nulls — will be filled with 'unknown'
- Remaining nulls are small edge cases

### Handle nulls
- Drop rows missing review_score (can't analyze satisfaction without it)
- Fill missing product_category_name with 'unknown'
- Drop remaining edge case nulls

In [14]:
# Drop rows with no review score
df = df.dropna(subset=['review_score'])

# Fill missing product category
df['product_category_name'] = df['product_category_name'].fillna('unknown')

# Drop remaining small nulls
df = df.dropna(subset=['order_delivered_customer_date', 'delivery_time_days', 'delay_days'])

print(df.shape)

(114854, 38)


### Save clean dataframe

In [15]:
output_path = "../outputs/"
df.to_csv(output_path + "olist_clean.csv", index=False)
print("Saved successfully")

Saved successfully
